# AIDP TPC-DS Benchmark — V1.2 Power Test (Portable)

Runs the official TPC-DS v4.0.0 benchmark on AIDP in **any** OCI tenancy — no hardcoded namespace, no private bucket fetch.

**Prerequisites (one-time setup):**
- AIDP Workbench cluster, Python 3.10+, Spark 3.5.x
- Cluster has internet egress to `pypi.org` (Cell 2 installs from public PyPI)
- IAM policy on the cluster's dynamic group — see `REQUIREMENTS.md`
- AIDP Unified Catalog is the default catalog for the Spark session

**To run:** set `SF` in Cell 1, then **Run All**. Total customer code: 1 line.

Version: `aidp_benchmark >= 0.1.2`

## Cell 1 — Configuration

The only knob a customer touches. `SF` is the TPC-DS scale factor (dataset size in GB). Start with `1` to validate end-to-end, then move to `10` or `100`.

In [ ]:
# CELL 1 — CONFIGURATION: change only this
SF = 10   # Scale factor: 1 | 10 | 100
print(f"Configured for SF={SF}")

## Cell 2 — Install `aidp-benchmark` (TEMP TEST PATH)

**TEMP test path:** fetches the wheel from an OCI bucket via Hadoop FS (V1.1 pattern, version 0.1.2). This is the staging path until the public distribution channel (PyPI vs GitHub release) is confirmed. The final V1.2 Cell 2 will be a clean `pip install aidp-benchmark>=0.1.2`.

In [ ]:
# CELL 2 — Install aidp-benchmark 0.1.2 (TEMP TEST PATH, V1.1 pattern)
# Fetches the wheel from OCI bucket via Hadoop FS, installs it without deps.
# Revert to `pip install aidp-benchmark` once distribution channel is confirmed.

import subprocess, sys, os, shutil

if os.path.isdir("/tmp/pkg2"):
    shutil.rmtree("/tmp/pkg2")
for k in list(sys.modules.keys()):
    if "aidp_benchmark" in k:
        del sys.modules[k]

os.makedirs("/tmp/dl", exist_ok=True)
os.chmod("/tmp/dl", 0o777)
os.makedirs("/tmp/pkg2", exist_ok=True)

from py4j.java_gateway import java_import
java_import(spark._jvm, 'org.apache.hadoop.fs.FileSystem')
java_import(spark._jvm, 'org.apache.hadoop.fs.Path')
fs = spark._jvm.FileSystem.get(
    spark._jvm.java.net.URI.create("oci://tpcds-benchmark-sf1@idseylbmv0mm/"),
    spark.sparkContext._jsc.hadoopConfiguration()
)
fs.copyToLocalFile(False,
    spark._jvm.Path("oci://tpcds-benchmark-sf1@idseylbmv0mm/aidp_benchmark-0.1.2-py3-none-any.whl"),
    spark._jvm.Path("/tmp/dl/aidp_benchmark-0.1.2-py3-none-any.whl"))
print("downloaded 0.1.2")

r = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "/tmp/dl/aidp_benchmark-0.1.2-py3-none-any.whl",
     "--target", "/tmp/pkg2", "--no-deps", "-q"],
    capture_output=True, text=True, cwd="/tmp")
if r.returncode != 0:
    raise RuntimeError(r.stderr[-300:])
print("installed aidp_benchmark 0.1.2")

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "duckdb",
     "--target", "/tmp/pkg2", "-q",
     "--index-url", "https://pypi.org/simple/",
     "--trusted-host", "pypi.org",
     "--trusted-host", "files.pythonhosted.org"],
    capture_output=True, text=True, cwd="/tmp")
if r.returncode != 0:
    raise RuntimeError(r.stderr[-300:])
print("installed duckdb")

sys.path.insert(0, "/tmp/pkg2")
import aidp_benchmark, duckdb
print(f"aidp_benchmark: {aidp_benchmark.__version__}   duckdb: {duckdb.__version__}")
assert aidp_benchmark.__version__ == "0.1.2", "wrong version installed"

## Cell 3 — Generate data + run the benchmark

Three things happen here, automatically:

1. **Auto-detect tenancy** — namespace, region, and compartment come from the cluster's Resource Principal. No hardcoded tenancy strings.
2. **Generate + upload data** — DuckDB `dsdgen` produces 24 Parquet files locally, then streams them to `oci://tpcds-benchmark-sf{SF}/`. Idempotent: re-runs skip tables already in the bucket.
3. **Run all 103 queries** — TPC-DS v4.0.0 with the official a/b splits. Sequential power test. GeoMean is the headline metric.

Expect ~10s GeoMean at `SF=10` on a default AIDP cluster (V1.1 baseline: 9.67s).

In [ ]:
# CELL 3 — Run the benchmark (namespace auto-detected via Resource Principal)
from aidp_benchmark import TPCDSDataGenerator, AIDPBenchmark

gen = TPCDSDataGenerator(scale_factor=SF)
gen_result = gen.generate(spark=spark)
print(f"Data ready: bucket={gen_result.bucket_name} "
      f"status={gen_result.bucket_status} "
      f"duration={gen_result.duration_seconds}s")
print(f"Upload: {gen_result.upload_summary}")

result = AIDPBenchmark(gen).run_power_test()
print(result.summary())

## Cell 4 — Save results to AIDP Unified Catalog

Writes two Iceberg tables in the `aidp_benchmark` catalog database:
- `aidp_benchmark.run_summary` — one row per benchmark run (GeoMean, P50/P95/P99, queries passed/failed)
- `aidp_benchmark.query_results` — one row per query per run (elapsed time, status, execution plan)

Append-only. Safe to re-run.

In [ ]:
# CELL 4 — Save to AIDP Unified Catalog (Iceberg)
result.save()
print(f"Saved run {result.run_id}")
print("  → aidp_benchmark.run_summary")
print("  → aidp_benchmark.query_results")

## Cell 5 — Verify results

Reads the run back out of the catalog as a sanity check. Shows the summary row + top 10 slowest queries.

In [ ]:
# CELL 5 — Verify: read back from catalog
print("=== run_summary ===")
spark.sql(f"""
    SELECT run_id, scale_factor, queries_passed, queries_failed,
           geomean_seconds, p95_seconds, end_time
    FROM aidp_benchmark.run_summary
    WHERE run_id = '{result.run_id}'
""").show(truncate=False)

print("=== top 10 slowest queries ===")
spark.sql(f"""
    SELECT query_name, elapsed_seconds, status
    FROM aidp_benchmark.query_results
    WHERE run_id = '{result.run_id}'
    ORDER BY elapsed_seconds DESC LIMIT 10
""").show(truncate=False)

## Cell 6 — Optional one-time cleanup (DISABLED BY DEFAULT)

Drops the entire `aidp_benchmark` catalog database. **Destructive — wipes ALL prior runs from `run_summary` AND `query_results`.** Uncomment only if you want a clean slate.

In [ ]:
# CELL 6 (OPTIONAL, ONE-TIME) — Wipe the aidp_benchmark catalog
# WARNING: deletes ALL prior benchmark runs. Uncomment ONLY for first-run cleanup.
#
# spark.sql("DROP DATABASE IF EXISTS aidp_benchmark CASCADE")
# print("Dropped — next run will recreate the database from scratch.")
print("Cell 6 skipped (wipe commented out — uncomment manually to use).")

---
**Done.** Compare results with `SELECT * FROM aidp_benchmark.run_summary ORDER BY end_time DESC LIMIT 5`. File issues at the project's GitHub repo.

> **Note:** Results produced by this toolkit are not comparable to official TPC Benchmark Results.